# Bakta CTS Demo

End-to-end test of the cdm_bakta CTS tool registered 2026-05-07.

- **Image:** `ghcr.io/kbaseincubator/cdm_bakta:0.1.0@sha256:6de4c51cadd75bc6a1d9f6e6b05716ecfdcfa63510b82459477ff757200d8d06`
- **Refdata UUID:** `663783c1-961a-492f-834f-4755914dc92a`
- **Refdata file:** `cts-refdata/bakta/v6.0/bakta_db.tar.gz` (45.9GB compressed, ~84GB unpacked)
- **Cluster:** `kbase`
- **Output:** `cts/io/jplfaria/output/bakta/test/v1`

Bakta does full bacterial genome annotation. Input is **nucleotide FASTA** (genome assemblies, `.fna` or `.fna.gz`), unlike kofamscan which needs protein sequences.

## 1. Setup

In [6]:
tscli = get_task_service_client()
mincli = get_minio_client()

IMAGE = "ghcr.io/kbaseincubator/cdm_bakta:0.1.1@sha256:31d2538fe5265fe4112c861f1edd1c2276b59ef8c16b9fda148c2b648e7c5266"
OUTPUT_DIR = "cts/io/jplfaria/output/bakta/test/v1"

print(tscli.whoami())

{'user': 'jplfaria', 'roles': [], 'allowed_paths': [{'path': 'cts/io/', 'perm': 'write'}]}


## 2. List input genomes

Use the same 4 test genomes the rest of the team has been using (`cts/io/gavin/test_files/`). These are nucleotide assemblies with CRC64NVME checksums.

In [7]:
input_files = []
for o in mincli.list_objects("cts", prefix="io/gavin/test_files", recursive=True):
    if o.object_name.endswith(".fna.gz") or o.object_name.endswith(".fna"):
        input_files.append(f"cts/{o.object_name}")

print(f"{len(input_files)} input genome(s):")
for f in input_files:
    print(f"  {f}")

4 input genome(s):
  cts/io/gavin/test_files/collections/NONE/CDM/FastGenomics/GCA_000008085.1/GCA_000008085.1_ASM808v1_genomic.fna.gz
  cts/io/gavin/test_files/collections/NONE/CDM/FastGenomics/GCA_000010565.1/GCA_000010565.1_ASM1056v1_genomic.fna.gz
  cts/io/gavin/test_files/collections/NONE/CDM/FastGenomics/GCA_000145985.1/GCA_000145985.1_ASM14598v1_genomic.fna.gz
  cts/io/gavin/test_files/collections/NONE/CDM/FastGenomics/GCA_000147015.1/GCA_000147015.1_ASM14701v1_genomic.fna.gz


## 3. Submit Bakta job

Bakta is heavier than kofamscan: full DB (~84GB unpacked) + lots of subprocess steps (CDS prediction, tRNA/rRNA scans, BLAST against UniRef, etc). Plan for ~20-40 min per bacterial genome at 4 CPUs.

One container per input genome (parallelizes the wall time).

In [8]:
if not input_files:
    raise RuntimeError("No .fna(.gz) inputs found. Adjust the prefix above.")

job = tscli.submit_job(
    IMAGE,
    input_files,
    OUTPUT_DIR,
    cluster="kbase",
    declobber=True,
    output_mount_point="/out",
    args=[
        "--db", "/ref_data/db",
        "--output", "/out",
        "--threads", "4",
        "--keep-contig-headers",
        "--skip-plot",        # skip the circular-genome PNG/SVG (slow + not needed for this test)
        "--force",            # bakta refuses to write to existing /out (CTS pre-creates it)
        tscli.insert_files(),
    ],
    num_containers=len(input_files),
    cpus=4,
    memory="32GB",
    runtime="PT4H",
)
print("Job ID:", job.id)

Job ID: b32fc4d4-e8ff-4a76-9e55-49c2f058487d


## 4. Monitor (non-blocking)

In [19]:
import threading, json
thread = threading.Thread(
    target=lambda: print(json.dumps(job.wait_for_completion(), indent=4)),
    daemon=True,
)
thread.start()

{
    "id": "b32fc4d4-e8ff-4a76-9e55-49c2f058487d",
    "state": "error",
    "transition_times": [
        {
            "state": "created",
            "time": "2026-05-12T17:54:32.653000Z"
        },
        {
            "state": "download_submitted",
            "time": "2026-05-12T17:54:32.718000Z"
        },
        {
            "state": "job_submitting",
            "time": "2026-05-12T17:55:01.487000Z"
        },
        {
            "state": "error",
            "time": "2026-05-12T17:55:10.459000Z"
        }
    ],
    "user": "jplfaria",
    "admin_meta": {}
}


In [20]:
# Re-run this cell to poll status
print(json.dumps(job.get_job_status(), indent=2, default=str))

{
  "id": "b32fc4d4-e8ff-4a76-9e55-49c2f058487d",
  "state": "error",
  "transition_times": [
    {
      "state": "created",
      "time": "2026-05-12T17:54:32.653000Z"
    },
    {
      "state": "download_submitted",
      "time": "2026-05-12T17:54:32.718000Z"
    },
    {
      "state": "job_submitting",
      "time": "2026-05-12T17:55:01.487000Z"
    },
    {
      "state": "error",
      "time": "2026-05-12T17:55:10.459000Z"
    }
  ],
  "user": "jplfaria",
  "admin_meta": {}
}


## 5. Inspect output files

In [21]:
outs = job.get_job()["outputs"]
print(f"{len(outs)} total output files")
for o in outs:
    print(f"  {o['file']}")

KeyError: 'outputs'

## 6. Read annotation TSVs as a dataframe

Bakta produces one `.tsv` per input genome with columns:
`Sequence Id, Type, Start, Stop, Strand, Locus Tag, Gene, Product, DbXrefs`.

Lines starting with `#` are headers/metadata; the actual table starts after a `#Sequence Id` header.

In [22]:
import io
import pandas as pd

tsvs = [o for o in outs if o["file"].endswith(".tsv") and "/tmp/" not in o["file"]]
print(f"{len(tsvs)} bakta .tsv annotation file(s)")

frames = []
for o in tsvs:
    bucket, key = o["file"].split("/", 1)
    obj = mincli.get_object(bucket, key)
    raw = obj.read().decode("utf-8")
    # Strip metadata comment lines, find the column header line, then parse the rest
    lines = raw.splitlines()
    header_idx = next((i for i, ln in enumerate(lines) if ln.startswith("#Sequence Id")), None)
    if header_idx is None:
        print(f"  WARN: no header line in {o['file']}")
        continue
    table = "\n".join([lines[header_idx][1:]] + lines[header_idx + 1:])
    df = pd.read_csv(io.StringIO(table), sep="\t")
    df["source_file"] = o["file"]
    frames.append(df)

all_df = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
print(f"\nTotal feature rows: {len(all_df)}")
if len(all_df):
    print("\nFeature counts by Type:")
    print(all_df["Type"].value_counts())
all_df.head(20)

NameError: name 'outs' is not defined

## 7. End-to-end check

If all of the below are True, the registration is fully working and we can close issue #2.

In [23]:
checks = {
    "job complete": job.get_job_status()["state"] == "complete",
    "has bakta .tsv files": len(tsvs) > 0,
    "non-empty annotations": len(all_df) > 0,
    "has CDS features": (all_df["Type"] == "cds").any() if len(all_df) else False,
    "has Product assignments": all_df["Product"].notna().any() if len(all_df) else False,
}
for k, v in checks.items():
    print(f"  [{'x' if v else ' '}] {k}")

if all(checks.values()):
    print("\nAll green. Safe to tick the final box on issue #2 and close.")
else:
    print("\nSomething's off. Don't close the issue yet.")

NameError: name 'tsvs' is not defined

## 8. Diagnostic (run if job errored)

In [24]:
# Error diagnostic - only run if state is error
ed = job.get_job()
print("logpath:", ed.get("logpath"))
print("error:", ed.get("error"))
print("exit codes:", job.get_exit_codes())
print("---stderr container 0---")
try:
    job.print_logs(container_num=0, stderr=True)
except Exception as e:
    print("stderr unavailable:", e)
print("---stdout container 0---")
try:
    job.print_logs(container_num=0, stderr=False)
except Exception as e:
    print("stdout unavailable:", e)


CTS returned error structure:
{'error': {'httpcode': 404, 'httpstatus': 'Not Found', 'time': '2026-05-12T18:48:49.926440+00:00', 'request_id': '3973dfd9-7634-4b27-8726-83097c679645', 'appcode': 40070, 'apperror': 'No logs available', 'message': 'Job ID b32fc4d4-e8ff-4a76-9e55-49c2f058487d has no logs available'}}


logpath: None
error: An unexpected error occurred.
exit codes: {'exit_codes': [None, None, None, None]}
---stderr container 0---
stderr unavailable: Job ID b32fc4d4-e8ff-4a76-9e55-49c2f058487d has no logs available
---stdout container 0---


CTS returned error structure:
{'error': {'httpcode': 404, 'httpstatus': 'Not Found', 'time': '2026-05-12T18:48:50.013463+00:00', 'request_id': '8fcd6995-0d22-414f-aca5-fd7153e0df74', 'appcode': 40070, 'apperror': 'No logs available', 'message': 'Job ID b32fc4d4-e8ff-4a76-9e55-49c2f058487d has no logs available'}}


stdout unavailable: Job ID b32fc4d4-e8ff-4a76-9e55-49c2f058487d has no logs available


## 6. Read annotations.tsv as a dataframe

`detail-tsv` columns: gene_name, KO, threshold, score, e_value, KO_definition.
Lines starting with `#` are headers/comments; first non-comment line is column names.

## 9. Check first job (sanity)

In [25]:
# Check the original (first) bakta job to see if it failed the same way
first = tscli.get_job_by_id("9d60b608-9448-4d36-abf3-c9c5ea891074")
import json
print(json.dumps(first.get_job_status(), indent=2, default=str))
print("exit codes:", first.get_exit_codes())


{
  "id": "9d60b608-9448-4d36-abf3-c9c5ea891074",
  "state": "error",
  "transition_times": [
    {
      "state": "created",
      "time": "2026-05-12T17:22:12.003000Z"
    },
    {
      "state": "download_submitted",
      "time": "2026-05-12T17:22:12.093000Z"
    },
    {
      "state": "job_submitting",
      "time": "2026-05-12T17:22:26.212000Z"
    },
    {
      "state": "error",
      "time": "2026-05-12T17:22:54.260000Z"
    }
  ],
  "user": "jplfaria",
  "admin_meta": {}
}
exit codes: {'exit_codes': [None, None, None, None]}
